# Tiny Recursive MoE Contrastive (TRMC) Model Training

This notebook demonstrates how to train the TRMC model on a synthetic reasoning task using both standard cross-entropy and contrastive learning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import sys
sys.path.append('..')

from trmc_model import TRMCModel, contrastive_loss
from adaptive_context import AdaptiveContextManager
from dataset_curator import TRMCDatasetCurator

In [ ]:
class LogicPuzzlesDataset(Dataset):
    def __init__(self, size=1000, seq_len=16, vocab_size=10, num_negatives=5):
        self.size = size
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.num_negatives = num_negatives
        self.data = []
        for _ in range(size):
            x = torch.randint(3, vocab_size, (seq_len,))
            y_pos = torch.flip(x, dims=[0])
            y_negs = []
            for _ in range(num_negatives):
                y_neg = y_pos.clone()
                idx = torch.randint(0, seq_len, (max(1, seq_len // 4),))
                y_neg[idx] = torch.randint(3, vocab_size, (len(idx),))
                y_negs.append(y_neg)
            self.data.append((x, y_pos, torch.stack(y_negs)))
    def __len__(self): return self.size
    def __getitem__(self, idx): return self.data[idx]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size, seq_len, hidden_dim = 32, 16, 128
dataset = LogicPuzzlesDataset(size=500, seq_len=seq_len, vocab_size=vocab_size)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = TRMCModel(
    vocab_size=vocab_size, 
    hidden_dim=hidden_dim, 
    num_experts=8, 
    num_iterations=8,
    matryoshka_dims=[32, 64, 128]
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
contrastive_weight = 0.1

for epoch in range(2):
    total_loss = 0
    for x, y_pos, y_negs in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        x, y_pos, y_negs = x.to(device), y_pos.to(device), y_negs.to(device)
        images = torch.randn(x.shape[0], 3, 32, 32).to(device)
        
        optimizer.zero_grad()
        logits, query_latent = model(x, images=images)
        
        with torch.no_grad():
            _, pos_latent = model(y_pos, images=images)
            batch_size, num_negs, s_len = y_negs.shape
            _, neg_latent_all = model(y_negs.view(-1, s_len), images=images.repeat_interleave(num_negs, dim=0))
            neg_latents = neg_latent_all.view(batch_size, num_negs, -1, hidden_dim)
            
        ce_loss = criterion(logits.view(-1, vocab_size), y_pos.view(-1))
        c_loss = contrastive_loss(
            query_latent.mean(1), 
            pos_latent.mean(1), 
            neg_latents.mean(2),
            matryoshka_dims=model.matryoshka_dims
        )
        loss = ce_loss + contrastive_weight * c_loss
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    print(f"Avg Loss: {total_loss/len(dataloader):.4f}")